Importación librerías y directorios

In [1]:
import pandas as pd
import os
import json
from dotenv import load_dotenv
import requests
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt

In [2]:
import sys
from pathlib import Path

for candidato in (Path.cwd(), *Path.cwd().resolve().parents):
    if (candidato / "src").is_dir():
        if str(candidato) not in sys.path:
            sys.path.insert(0, str(candidato))
        break

from src.paths import RAIZ, DIR_RAW, DIR_PROCESSED, DIR_REPORTS
from src.esios_client import crear_sesion, descargar_indicador, descargar_rango

Descarga de token

In [3]:
load_dotenv()
# Si tu .env está en una carpeta superior o una ruta específica, puedes indicarla:
# load_dotenv(dotenv_path="../.env")

# Recuperamos el token de forma segura
TOKEN = os.getenv("API_ESIOS")

# Verificación rápida (sin mostrar el token entero por seguridad)
if TOKEN:
    print(f"✅ Token cargado correctamente. Longitud: {len(TOKEN)} caracteres.")
else:
    print("❌ No se pudo encontrar la variable ESIOS_TOKEN. Revisa la ruta del archivo .env.")

✅ Token cargado correctamente. Longitud: 64 caracteres.


Descarga de datos

In [4]:
sesion = crear_sesion(TOKEN) 

resumen_1293 = descargar_rango(sesion, 1293, '2023-01-01', '2026-07-01', 'demanda_real')
resumen_544  = descargar_rango(sesion, 544,  '2023-01-01', '2026-07-01', 'demanda_prevista')

reporte = pd.DataFrame(resumen_1293 + resumen_544)
reporte['estado'].value_counts()

estado
EXISTENTE (SKIPPED)    84
Name: count, dtype: int64

Comprobación

In [6]:
import shutil, pandas as pd
from src.paths import DIR_RAW

for candidato in (Path.cwd(), *Path.cwd().resolve().parents):
    if (candidato / "src").is_dir():
        if str(candidato) not in sys.path:
            sys.path.insert(0, str(candidato))
        break

import shutil
import pandas as pd
from src.paths import DIR_RAW
from src.esios_client import crear_sesion, descargar_rango

# 1. Aparta los seis de 2026 SIN borrarlos
respaldo = DIR_RAW.parent / "raw_backup_2026"
respaldo.mkdir(exist_ok=True)
for f in sorted(DIR_RAW.glob("1293_2026-*.parquet")):
    shutil.move(str(f), respaldo / f.name)

import os
from dotenv import load_dotenv
load_dotenv()
sesion = crear_sesion(os.getenv("API_ESIOS"))

# 2. Redescarga: al no existir el fichero, el skip no aplica
resumen = descargar_rango(sesion, 1293, '2026-01-01', '2026-07-01', 'demanda_real')
print(pd.DataFrame(resumen)[['mes','filas_obtenidas','estado']])

# 3. Compara viejo contra nuevo, mes a mes
for f_new in sorted(DIR_RAW.glob("1293_2026-*.parquet")):
    a = pd.read_parquet(respaldo / f_new.name)["demanda_real"]
    b = pd.read_parquet(f_new)["demanda_real"]
    print(f_new.stem, f"media vieja {a.mean():.0f}  nueva {b.mean():.0f}  dif {b.mean()-a.mean():+.0f}")

       mes  filas_obtenidas estado
0  2026-01             8928     OK
1  2026-02             8064     OK
2  2026-03             8928     OK
3  2026-04             8640     OK
4  2026-05             8928     OK
5  2026-06             8640     OK
1293_2026-01 media vieja 30719  nueva 30719  dif +0
1293_2026-02 media vieja 29415  nueva 29415  dif +0
1293_2026-03 media vieja 28456  nueva 28456  dif +0
1293_2026-04 media vieja 26498  nueva 26498  dif +0
1293_2026-05 media vieja 27156  nueva 27156  dif +0
1293_2026-06 media vieja 30933  nueva 30933  dif +0


In [7]:
for fecha in ["2025-06-15", "2026-06-15"]:
    r = sesion.get("https://api.esios.ree.es/indicators/1293",
                   params={"start_date": f"{fecha}T00:00", "end_date": f"{fecha}T01:00"},
                   timeout=(5, 60))
    vals = r.json()["indicator"]["values"]
    print(fecha, len(vals))
    print(" claves:", list(vals[0].keys()))
    print(" geos:", {(v.get("geo_id"), v.get("geo_name")) for v in vals})
    

2025-06-15 13
 claves: ['value', 'datetime', 'datetime_utc', 'tz_time', 'geo_id', 'geo_name']
 geos: {(8741, 'Península')}
2026-06-15 13
 claves: ['value', 'datetime', 'datetime_utc', 'tz_time', 'geo_id', 'geo_name']
 geos: {(8741, 'Península')}
